# Train / test split info (matches `trainer.py`)

Uses the same settings as training:

- Same split as `trainer.py`: `torch.Generator().manual_seed(42)` passed to `random_split` (not global `manual_seed` + `randperm`, so the index list matches exactly)
- `validation_split = 0.2` → first `train_size` indices = **train**, remaining = **test** (validation holdout)
- CSV path from `multi-class-classifier/config.py` (`transformer_preprocessed.csv`)

**Presence** for each element in `train_agg` / `test_agg`: label ≠ 0 (absent) in the preprocessed integer columns.

**Subelement counts**: per head, counts for each class label `0 … num_classes-1` in train vs test (`class_counts_long`, wide tables, text summary).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import TensorDataset, random_split

REPO = Path.cwd().resolve()
MCC = REPO / "multi-class-classifier"
if MCC.is_dir():
    sys.path.insert(0, str(MCC))

from config import CSV_FILE, HEADS

SEED = 42
VALIDATION_SPLIT = 0.2

csv_path = MCC / CSV_FILE if (MCC / CSV_FILE).is_file() else REPO / CSV_FILE
if not csv_path.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

df = pd.read_csv(csv_path)
n = len(df)
val_size = int(n * VALIDATION_SPLIT)
train_size = n - val_size

idx_ds = TensorDataset(torch.arange(n, dtype=torch.long))
split_generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(idx_ds, [train_size, val_size], generator=split_generator)
train_idx = list(train_ds.indices)
test_idx = list(val_ds.indices)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"CSV: {csv_path}")
print(f"Seed: {SEED}, validation_split: {VALIDATION_SPLIT}")
print(f"Total rows: {n} | train: {len(train_df)} | test (val holdout): {len(test_df)}")

In [ ]:
label_cols = [name for name, _ in HEADS]


def element_aggregates(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    total = len(frame)
    for col in label_cols:
        if col not in frame.columns:
            raise KeyError(f"Missing column {col} in CSV")
        s = frame[col]
        if pd.api.types.is_numeric_dtype(s):
            present = s.fillna(0).astype(int).ne(0)
        else:
            present = s.fillna("").astype(str).str.strip().ne("")
        n_present = int(present.sum())
        rows.append(
            {
                "split": split_name,
                "model_column": col,
                "n_present": n_present,
                "n_absent": total - n_present,
                "n_total": total,
                "present_rate": n_present / total if total else 0.0,
            }
        )
    return pd.DataFrame(rows)


def per_class_long(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """One row per (element, label) with count. Labels 0 .. num_classes-1 from config."""
    rows = []
    for col, num_classes in HEADS:
        if col not in frame.columns:
            raise KeyError(col)
        s = pd.to_numeric(frame[col], errors="coerce").fillna(0).astype(int)
        bad = s[(s < 0) | (s >= num_classes)]
        if len(bad):
            print(f"Warning [{split_name}] {col}: {len(bad)} values outside [0, {num_classes - 1}]")
        for lbl in range(num_classes):
            cnt = int((s == lbl).sum())
            rows.append(
                {
                    "split": split_name,
                    "model_column": col,
                    "num_classes": num_classes,
                    "label": lbl,
                    "count": cnt,
                }
            )
    return pd.DataFrame(rows)


def pivot_class_wide(class_long: pd.DataFrame, split_name: str) -> pd.DataFrame:
    sub = class_long[class_long["split"] == split_name].copy()
    wide = sub.pivot_table(
        index=["model_column", "num_classes"],
        columns="label",
        values="count",
        aggfunc="sum",
    ).fillna(0).astype(int)
    wide.columns = [f"label_{int(c)}" for c in wide.columns]
    return wide.reset_index()


train_agg = element_aggregates(train_df, "train")
test_agg = element_aggregates(test_df, "test")

train_class_long = per_class_long(train_df, "train")
test_class_long = per_class_long(test_df, "test")
class_counts_long = pd.concat([train_class_long, test_class_long], ignore_index=True)

train_class_wide = pivot_class_wide(class_counts_long, "train")
test_class_wide = pivot_class_wide(class_counts_long, "test")


def print_element_breakdown(class_long: pd.DataFrame, split_name: str) -> None:
    sub = class_long[class_long["split"] == split_name]
    n_split = len(train_df) if split_name == "train" else len(test_df)
    for col, num_classes in HEADS:
        g = sub[sub["model_column"] == col].sort_values("label")
        if g.empty:
            continue
        assert int(g["count"].sum()) == n_split
        parts = ", ".join(f"label {int(r.label)}: {int(r.count)}" for r in g.itertuples())
        print(f"{col} — {n_split} rows in split | {parts}")
    print()

## Train / test presence summary (`train_agg`, `test_agg`)

In [ ]:
train_agg

In [ ]:
test_agg

## Per-class (subelement) counts — long table

`class_counts_long`: one row per split × head × label (`0` … `num_classes-1`).

In [ ]:
class_counts_long.sort_values(["split", "model_column", "label"]).reset_index(drop=True)

## Per-class counts — wide tables (`label_0`, `label_1`, …)

In [ ]:
from IPython.display import display

print("train_class_wide")
display(train_class_wide)
print("\ntest_class_wide")
display(test_class_wide)

## Text summary (one line per head)

In [ ]:
print("=== TRAIN ===")
print_element_breakdown(class_counts_long, "train")
print("=== TEST (val holdout) ===")
print_element_breakdown(class_counts_long, "test")